# Lane 3: Energy-Search - Google Colab Runner
**Project:** Lyapunov Residual Stream Pruning for Reasoning  
**Author:** Jaime Jarauta  
**Pre-Registered Question:** Can model-intrinsic dynamical instability metrics detect reasoning derailment and prune bad rollout branches without an external PRM?

Optimized for Google Colab T4 and local GPUs with KV-caching.


In [ ]:
# Step 1: Pre-flight Hardware Verification
import sys, os, time, math
import torch

print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    device = torch.device("cuda")
    dev_name = torch.cuda.get_device_name(0)
    print(f"Device: {dev_name}")
    torch.set_float32_matmul_precision("high")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Device: Apple Silicon MPS")
else:
    device = torch.device("cpu")
    print("Device: CPU")


In [ ]:
# Step 2: Dependencies & Imports
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q uv
    !uv pip install --system -q transformers pyyaml wandb pytest

if "src" not in sys.path:
    sys.path.insert(0, "src")

from energy_search.sampler import sample_with_energy_pruning
from energy_search.pruner import DynamicalInstabilityPruner
from energy_search.telemetry import compute_relative_velocity, compute_token_entropy

print("Energy-Search modules loaded successfully.")


In [ ]:
# Step 3: Run KV-Cached Pruning Smoke Test
import torch.nn as nn

class ToyTransformerLM(nn.Module):
    def __init__(self, vocab_size=100, d_model=64, n_layers=4):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(d_model=d_model, nhead=4, dim_feedforward=128, batch_first=True)
            for _ in range(n_layers)
        ])
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, token_ids):
        h = self.embed(token_ids)
        for layer in self.layers:
            h = layer(h)
        logits = self.head(h)
        return logits, h

model = ToyTransformerLM().to(device)
model.eval()

prompt = torch.randint(0, 100, (1, 10), device=device)
t0 = time.time()
res = sample_with_energy_pruning(
    model=model,
    input_ids=prompt,
    max_new_tokens=40,
    tau=2.5,
    k_persist=4,
    freeze_vel_tau=0.05,
    freeze_ent_tau=0.15,
    k_freeze=6,
    device=device
)
elapsed = time.time() - t0
latency_ms = (elapsed / max(1, res['tokens_generated'])) * 1000.0

print(f"Tokens Generated: {res['tokens_generated']} | Time: {elapsed:.3f}s")
print(f"Latency per Token: {latency_ms:.2f} ms")
print(f"Pruned: {res['pruned']} (Step: {res['pruned_at_step']}, Reason: {res['prune_reason']})")
print(f"Mean Instability Index: {res['mean_instability']:.4f}")


In [ ]:
# Step 4: Pre-Registered Gate Assessment
passed = latency_ms < 5.0
print("=" * 60)
print(f"Telemetry Latency: {latency_ms:.2f} ms/token (Target: < 5.0 ms)")
print(f"Pre-Registered Gate: {'PASSED (Low overhead telemetry)' if passed else 'FAILED (Kill)'}")
print("=" * 60)
